In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path
from collections import defaultdict

from utils.rocksdb_parser import RocksDBParser
from utils.plot import *

In [ ]:
NUM_CORES_LIST = [1, 2, 4, 8, 16]
LATENCY_LIST = [0, 500, 1000, 2000, 3000, 4000, 5000, 10000]

In [ ]:
parser = RocksDBParser('../rocksdb/log')

In [ ]:
NUM_IOS_PER_OP = 0.333

SWITCH_TIME = 50
NUM_PREFETCHES = 12
NUM_CHASES = 10.21 / NUM_IOS_PER_OP
MEMORY_TIME = 230.0
IO_TIME_PRE = 8935.4
IO_TIME_POST = 369.5
TITLE = 'RocksDB (1B items, single core, read-only)'
NAME = 'rocksdb1b_1core_rdonly'


throughputs = []
for latency in LATENCY_LIST:
    t = parser.get_max_throughput('readrandom', num_cores=1, latency=latency)
    throughputs.append(t)

_ = plot_with_models(LATENCY_LIST, throughputs,
                     NUM_CHASES, MEMORY_TIME, IO_TIME_PRE, IO_TIME_POST,
                     SWITCH_TIME, NUM_PREFETCHES, 'RocksDB', TITLE, NAME)

In [ ]:
throughput_dict = defaultdict(list)
for i, lat in enumerate(LATENCY_LIST):
    for num_cores in NUM_CORES_LIST:
        t = parser.get_max_throughput('readrandom', num_cores, lat)
        throughput_dict[lat].append(t)

with open('rocksdb.json', 'w') as f:
    json.dump(throughput_dict, f, indent=4)

plot_core_scaling(NUM_CORES_LIST, throughput_dict, 'RocksDB (1B items, read-only)')

In [ ]:
throughputs = []
for latency in LATENCY_LIST:
    t = parser.get_max_throughput('readrandom', num_cores=16, latency=latency)
    throughputs.append(t)

plot_throughputs(LATENCY_LIST, throughputs, 2.1, 'C2',
                 'RocksDB (1B items, read-only)', 'rocksdb1b_16core_rdonly')

In [ ]:
throughputs = []
for latency in LATENCY_LIST:
    t = parser.get_max_throughput('readrandomwriterandom', num_cores=16, latency=latency)
    throughputs.append(t)

plot_throughputs(LATENCY_LIST, throughputs, 0.50, 'C2',
                 'RocksDB (1B items, read-write-mix)', 'rocksdb1b_16core')

In [ ]:
op_latencies = []
for latency in LATENCY_LIST:
    t = parser.get_perf('readrandomwriterandom', num_cores=16, latency=latency, num_threads=2048)
    op_latencies.append(t)

plot_latencies(LATENCY_LIST, op_latencies, [50, 90, 99], 1,
               'RocksDB (1B items, read-write mix)', 'rocksdb1b_16core_latency')